# Achilles analysis

Copy this notebook to start a new analysis (File > Save Notebook As).

`RUN_DIR` is the run this notebook server was started on (`icer-notebook <run id>`), the newest by default.
Other runs are under `runs/`. Plots saved to `RUN_DIR/plots` are copied to the laptop when this notebook
runs as a pipeline analysis job:

```
achilles-submit -a 'run-notebook ~/analysis/<name>.ipynb' run.yml
```

In [ ]:
import os, gzip
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

RUN_DIR = Path(os.environ.get("RUN_DIR", "."))
EVENT_FILES = sorted(os.environ.get("EVENT_FILES", "").split()) or sorted(
    str(p) for p in RUN_DIR.glob("task-*/*.hepmc*"))
PLOTS = RUN_DIR / "plots"
PLOTS.mkdir(exist_ok=True)
print(RUN_DIR)
print(len(EVENT_FILES), "event files")
print((RUN_DIR / "provenance.txt").read_text())

In [ ]:
def read_hepmc(files):
    """Yield each event in HepMC3 ASCII files as a dict:
    weight, and particles as an array with columns pdg, status, px, py, pz, E, m."""
    for name in files:
        opener = gzip.open if name.endswith(".gz") else open
        with opener(name, "rt") as f:
            event = None
            for line in f:
                tag = line[:2]
                if tag == "E ":
                    if event:
                        event["particles"] = np.array(event["particles"]).reshape(-1, 7)
                        yield event
                    event = {"weight": 1.0, "particles": []}
                elif tag == "W " and event is not None:
                    event["weight"] = float(line.split()[1])
                elif tag == "P " and event is not None:
                    p = line.split()
                    event["particles"].append([int(p[3]), int(p[9]), *map(float, p[4:9])])
            if event:
                event["particles"] = np.array(event["particles"]).reshape(-1, 7)
                yield event

events = list(read_hepmc(EVENT_FILES))
print(len(events), "events")

In [ ]:
# Example: energy of outgoing (status 1) electrons, weighted.
energy, weight = [], []
for ev in events:
    p = ev["particles"]
    sel = (p[:, 0] == 11) & (p[:, 1] == 1)
    energy += list(p[sel, 5])
    weight += [ev["weight"]] * int(sel.sum())

fig, ax = plt.subplots()
ax.hist(energy, bins=30, weights=weight)
ax.set_xlabel("Outgoing electron energy [MeV]")
ax.set_ylabel("Weighted events")
fig.savefig(PLOTS / "electron_energy.png", dpi=120)